# 13 — Verificación final
Chequeos independientes sobre lo que produjeron los pasos 01-12. Si alguno falla, el `assert`
detiene el notebook y dice cuál. Cada bloque cierra con "OK".

In [1]:
%run -i modulos/comun.ipynb
%run -i modulos/geografia.ipynb
%run -i modulos/materias.ipynb
%run -i modulos/juzgados.ipynb
%run -i modulos/contexto_procesos.ipynb
%run -i modulos/correcciones_tipo_proceso.ipynb
import numpy as np
from scipy.stats import spearmanr

AUDITORIA = PROCESSED / "auditoria"
ANALITICO = PROCESSED / "analitico"
TABLAS_PROCESOS = ["causas_por_tipo_proceso", "resueltas_por_tipo_proceso", "apelaciones_por_tipo_proceso",
                   "ejecucion_por_tipo_proceso", "otros_tramites_por_tipo_proceso"]


def tabla(nombre):
    return pd.read_parquet(PROCESSED / (nombre + ".parquet"))


def analitica(nombre):
    return pd.read_parquet(ANALITICO / (nombre + ".parquet"))


def vacio_a_none(texto):
    if texto == "":
        return None
    return texto


resultados = []


def ok(bloque):
    resultados.append(bloque)
    print("OK", bloque)

## 1. Las doce tablas y el CSV frente al Parquet

In [2]:
FILAS = {"causas_movimiento": 78, "causas_por_gestion": 235, "causas_serie_historica": 51, "juzgados": 1148,
         "personal": 85, "personal_jurisdiccional": 3, "autoridad_sumariante": 27, "causas_por_tipo_proceso": 2419,
         "resueltas_por_tipo_proceso": 27993, "apelaciones_por_tipo_proceso": 37871,
         "ejecucion_por_tipo_proceso": 10015, "otros_tramites_por_tipo_proceso": 6028}
total = 0
for nombre in FILAS:
    df = tabla(nombre)
    assert len(df) == FILAS[nombre], nombre
    csv = pd.read_csv(PROCESSED / (nombre + ".csv"), low_memory=False)
    assert list(csv.columns) == list(df.columns), nombre
    assert len(csv) == len(df), nombre
    total = total + len(df)
assert total == 85953
ok("doce tablas, 85.953 filas, CSV y Parquet con las mismas columnas")

OK doce tablas, 85.953 filas, CSV y Parquet con las mismas columnas


## 2. Geografía

In [3]:
assert departamento_de_ciudad("EL ALTO") == "La Paz"
assert departamento_de_ciudad("  cochabamba  ") == "Cochabamba"
assert departamento_de_ciudad("pOtOsÍ") == "Potosí"
for faltante in [None, np.nan, pd.NA, "", "   "]:
    assert normalizar_geografia(faltante) is None
assert departamento_segun_ambito("provincia", ciudad=pd.NA, distrito="  LA PAZ ") == "La Paz"
assert departamento_segun_ambito("capital", ciudad=None, distrito=None) is None
assert ambito_de_contexto("Juzgados de Instrucción Penal de Ciudades Capitales y El Alto", ["SUCRE"]) == "capital"
assert ambito_de_contexto("Juzgados de Instrucción Penal de Provincias", ["CHUQUISACA"]) == "provincia"
assert total_corresponde_a_entidad("5.3.2.1", 362, "capital", "LA PAZ", "TOTAL LA PAZ")
assert not total_corresponde_a_entidad("5.3.2.1", 362, "capital", "LA PAZ", "TOTAL EL ALTO")
assert total_corresponde_a_entidad("6.1.1.2", 409, "provincia", "CHUQUISACA", "TOTAL SUCRE")
assert not total_corresponde_a_entidad("5.3.1.3", 355, "capital", "TRINIDAD", "TOTAL BENI")

for nombre in TABLAS_PROCESOS:
    df = tabla(nombre)
    territorial = ~df["es_total_nacional"]
    assert df.loc[territorial, "departamento_derivado"].notna().all(), nombre
    assert set(df["departamento_derivado"].dropna()) <= set(DEPARTAMENTOS), nombre

# 6.3.1.4 de las páginas 357-359 es del capítulo 6 por número, pero de capitales por su encabezado.
resueltas = tabla("resueltas_por_tipo_proceso")
filas = resueltas[(resueltas["cuadro_origen"] == "6.3.1.4") & resueltas["pagina_pdf"].isin([357, 358, 359])]
assert len(filas) == 528 and set(filas["ambito"]) == {"capital"}
assert filas.loc[~filas["es_total_nacional"], "ciudad"].notna().all()
causas = tabla("causas_por_tipo_proceso")
p362 = causas[causas["pagina_pdf"] == 362]
assert p362["entidad"].value_counts().to_dict() == {"SUCRE": 10, "LA PAZ": 10, "EL ALTO": 10, "COCHABAMBA": 10}
assert len(pd.read_csv(AUDITORIA / "inconsistencias_geografia.csv")) == 0
ok("geografía")

OK geografía


## 3. Materias

In [4]:
propuesta = pd.read_csv(AUDITORIA / "propuesta_equivalencias_materias.csv", dtype=str).fillna("")
aprobadas = {}
for i, f in propuesta.iterrows():
    if f["decision"] in {"equivalente_confirmada", "equivalente_variacion_editorial"} and f["confianza"] == "alta":
        aprobadas[corregir_errata(f["materia_variante"])] = f["materia_canonica_propuesta"]
assert MATERIAS_HOMOLOGADAS == aprobadas

d = describir_materia("INSTRUCCÓN CONTRA LA VIOLENCIA HACIA LA MUJER")
assert d["materia_cruda"] == "INSTRUCCÓN CONTRA LA VIOLENCIA HACIA LA MUJER"
assert d["materia_norm"] == "INSTRUCCIÓN CONTRA LA VIOLENCIA HACIA LA MUJER"
assert d["errata_corregida"]

INDETERMINADAS = {"SENTENCIA ANTICORRUPCIÓN", "Sentencia Anticorrupción", "SENTENCIA CONTRA LA VIOLENCIA HACIA LA MUJER",
                  "Sentencia Violencia C M.", "Sentencia Violencia Contra la Violencia hacia las Mujeres",
                  "TRIBUNAL DE SENTENCIA ANTICORRUPCIÓN", "Tribunales de Sentencia Anticorrupción",
                  "TRIBUNAL DE SENTENCIA CONTRA LA VIOLENCIA HACIA LA MUJER",
                  "Tribunales de Sentencia Contra la Violencia hacia las Mujeres"}
observadas = set()
for nombre in ["causas_movimiento", "causas_por_gestion"] + TABLAS_PROCESOS:
    df = tabla(nombre)
    esperada = df["materia_norm"].map(homologar_materia)
    assert ((df["materia_homologada"] == esperada) | (df["materia_homologada"].isna() & esperada.isna())).all(), nombre
    f = df[df["materia_norm"].isin(INDETERMINADAS)]
    observadas = observadas | set(f["materia_norm"])
    assert (f["materia_homologada"] == f["materia_norm"]).all()
assert observadas == INDETERMINADAS

mov = tabla("causas_movimiento")
mov = mov[(mov["eje"] == "materia") & (mov["tipo_fila_derivado"] == "dato")]
ges = tabla("causas_por_gestion")
ges = ges[(ges["gestion"] == 2023) & (ges["tipo_fila_derivado"] == "dato")]
assert set(mov["materia_norm"]).isdisjoint(ges["materia_norm"])
assert len(set(mov["materia_homologada"]) & set(ges["materia_homologada"])) == 11
ok("materias: 11 equivalencias aprobadas y 9 variantes indeterminadas sin fusionar")

OK materias: 11 equivalencias aprobadas y 9 variantes indeterminadas sin fusionar


## 4. Juzgados

In [5]:
enc = pd.read_csv(AUDITORIA / "propuesta_encabezados_juzgados.csv", dtype=str).fillna("")
esperado = {}
for i, f in enc.iterrows():
    esperado[(f["cuadro_origen"], f["columna"])] = {"rotulo_canonico": vacio_a_none(f["rotulo_canonico"]),
                                                    "codigo_canonico": vacio_a_none(f["codigo_canonico"]),
                                                    "tipo_columna": f["tipo_columna"], "decision": f["decision"],
                                                    "confianza": f["confianza"]}
assert len(esperado) == 130 and ENCABEZADOS_JUZGADOS == esperado

filas_aud = pd.read_csv(AUDITORIA / "propuesta_filas_4_1_1.csv", dtype=str).fillna("")
esperado = {}
for i, f in filas_aud.iterrows():
    esperado[int(f["fila_orden"])] = {"fila_id": f["fila_id"], "literal_original": f["literal_original"],
                                      "rotulo_canonico": f["rotulo_canonico"], "codigo_canonico": f["codigo_canonico"],
                                      "tipo_entidad": f["tipo_entidad"], "estructura": f["estructura"],
                                      "fila_padre_id": vacio_a_none(f["fila_padre_id"]),
                                      "nivel_jerarquia": int(f["nivel_jerarquia"]), "decision": f["decision"],
                                      "confianza": f["confianza"]}
assert len(esperado) == 37 and FILAS_4_1_1 == esperado

juz = tabla("juzgados")
assert len(juz) == 1148
tarija = juz[(juz["cuadro_origen"] == "4.1.7") & (juz["columna"] == "col_09")]
assert len(tarija) == 3 and (tarija["tipo_columna"] == "indeterminado").all()
total_411 = juz[(juz["cuadro_origen"] == "4.1.1") & (juz["columna"] == "col_11")]
assert int(total_411["valor"].sum()) == 2422
assert int(total_411[total_411["es_hoja_jerarquia"] == True]["valor"].sum()) == 846
assert (pd.read_csv(AUDITORIA / "validacion_juzgados.csv")["estado"] == "OK").all()
ok("juzgados: 130 encabezados, 37 filas del 4.1.1, doble conteo 2.422 vs 846")

OK juzgados: 130 encabezados, 37 filas del 4.1.1, doble conteo 2.422 vs 846


## 5. Contexto y correcciones de tipo de proceso

In [6]:
etapas = pd.read_csv(AUDITORIA / "propuesta_etapas_tipo_proceso.csv", dtype=str)
assert len(etapas) == 18
mapa = {}
for i, f in etapas.iterrows():
    mapa[f["cuadro_origen"]] = f["etapa_proceso_fuente"]
assert mapa == ETAPA_AUDITADA_POR_CUADRO
contextos = pd.read_csv(AUDITORIA / "propuesta_contexto_accion_penal.csv")
assert len(contextos) == 54
assert set(contextos.loc[contextos["estructura_observada"].str.startswith("Una fila"), "cuadro_origen"]) == CUADROS_CONTEXTO_DIRECTO
assert set(contextos.loc[contextos["estructura_observada"].str.startswith("Celda padre"), "cuadro_origen"]) == CUADROS_CONTEXTO_BLOQUES
assert len(CUADROS_CONOCIDOS) == 95 and len(CUADROS_NO_APLICA) == 89

completa = pd.read_csv(AUDITORIA / "auditoria_fragmentos_tipo_proceso_completa.csv")
apariciones = {}
for i, f in completa.iterrows():
    apariciones[f["id_fragmento"].replace("F", "R", 1)] = int(f["apariciones_fuente"])
propuesta = pd.read_csv(AUDITORIA / "propuesta_correcciones_extraccion_tipo_proceso.csv", dtype=str)
assert len(propuesta) == 87
for i, f in propuesta.iterrows():
    r = CORRECCIONES_TIPO_PROCESO[i]
    paginas_regla = re.search(r"(?:^|\|)paginas=([^|]+)", f["clave_contexto"]).group(1)
    assert r["regla_id"] == f["regla_id"] and r["tabla"] == f["tabla"]
    assert r["cuadros"] == f["cuadro_origen"].split(";")
    assert r["paginas"] == [int(p) for p in paginas_regla.split(";")]
    assert r["literal_extraido"] == f["literal_actual"] and r["literal_fuente"] == f["literal_fuente_correcto"]
    assert r["alcance"] == f["alcance_correccion"] and r["apariciones_fuente"] == apariciones[f["regla_id"]]

dominio = set()
for nombre in TABLAS_PROCESOS:
    df = tabla(nombre)
    assert df["tipo_proceso_extraido"].nunique() == DOMINIOS_ANTES[nombre]
    assert df["tipo_proceso"].nunique() == DOMINIOS_DESPUES[nombre]
    dominio = dominio | set(df["tipo_proceso"].dropna())
    fuente = df.groupby(["cuadro_origen", "pagina_pdf", "orden_fila"], dropna=False)
    assert (fuente["etapa_proceso_fuente"].nunique(dropna=False) == 1).all()
    assert (fuente["contexto_accion_penal"].nunique(dropna=False) == 1).all()
assert len(dominio) == 147
assert (pd.read_csv(AUDITORIA / "validacion_contexto_procesos.csv")["estado"] == "OK").all()
assert (pd.read_csv(AUDITORIA / "validacion_correcciones_tipo_proceso.csv")["estado"] == "OK").all()
ok("contexto (18 etapas, 54 contextos) y 87 correcciones iguales a las auditorías")

OK contexto (18 etapas, 54 contextos) y 87 correcciones iguales a las auditorías


## 6. Integración interna

In [7]:
base = analitica("dataset_analitico_interno")
clave = ["ambito", "territorio", "materia_homologada", "tipo_proceso", "etapa_proceso_fuente", "contexto_accion_penal"]
assert len(base) == 1997 and len(base.columns) == 87
assert base.groupby(clave, dropna=False).ngroups == 1997
assert base["tipo_elemento_analitico"].value_counts().to_dict() == {"proceso": 1655, "accion_penal": 171, "otro_detalle": 171}
sumas = {}
for m in ["nuevas_ingresadas", "atendidas", "resueltas", "pendientes_fin", "pendientes_inicio"]:
    sumas[m] = int(base[m].sum())
assert sumas == {"nuevas_ingresadas": 378684, "atendidas": 706485, "resueltas": 212499, "pendientes_fin": 311478, "pendientes_inicio": 252037}
largo = analitica("metricas_tipo_proceso_long")
assert largo["familia_metrica"].value_counts().to_dict() == {"apelaciones": 37871, "resueltas": 27993, "ejecucion": 10015, "otros_tramites": 6028}
recursos = analitica("recursos_judiciales_geografia")
assert len(recursos) == 19 and "numero_juzgados_real" not in recursos.columns
tarija = recursos[(recursos["ambito"] == "provincia") & (recursos["territorio"] == "TARIJA")].iloc[0]
assert not bool(tarija["clasificacion_recursos_completa"]) and tarija["valor_indeterminado_publicado"] == 4
assert len(analitica("personal_geografia")) == 9
mg = analitica("movimiento_gestion_2023")
assert mg["estado_union"].value_counts().to_dict() == {"both": 32, "solo_movimiento": 12, "solo_gestion": 12}
assert len(pd.read_csv(ANALITICO / "diccionario_analitico.csv")) == 192
assert (pd.read_csv(AUDITORIA / "validacion_integracion_interna.csv")["estado"] == "OK").all()
ok("integración: 1.997 filas, clave única, uniones N:1 sin fan-out")

OK integración: 1.997 filas, clave única, uniones N:1 sin fan-out


## 7. Capa analítica (indicadores y outliers)

In [8]:
curado = pd.read_parquet(CURATED / "features" / "dataset_analitico_curado.parquet")
assert len(curado) == 1997 and len(curado.columns) == 118
proc = curado[curado["tipo_elemento_analitico"] == "proceso"]
v = proc.dropna(subset=["atendidas", "resueltas", "pendientes_fin"])
assert ((v["atendidas"] - (v["resueltas"] + v["pendientes_fin"])).abs() > 0).sum() <= 1
c = proc[proc["pendientes_inicio_efectivo"].notna() & proc["atendidas"].notna()]
assert len(c) == len(proc) == 1655, "la identidad de ingresos debe poder comprobarse en los 1.655 procesos"
assert ((c["atendidas"].astype(float) - c["pendientes_inicio_efectivo"].astype(float)) == c["ingresos_totales"].astype(float)).all()
# En civil el stock inicial es readecuadas_ley_439 (no un ingreso); en el resto, el pendientes_inicio publicado.
es_civil = proc["materia_homologada"].str.contains("Civil")
assert (proc.loc[es_civil, "fuente_stock_inicial"] == "readecuadas_ley_439").all() and int(es_civil.sum()) == 475
assert (proc.loc[~es_civil, "fuente_stock_inicial"] == "publicado").all()
assert int(proc.loc[es_civil, "pendientes_inicio_efectivo"].sum()) == 43062
# El cálculo anterior sigue siendo reproducible con su nombre honesto: 209.197 / 250.871 = 83,39 %.
anterior = proc["resueltas"].sum() / (proc["ingresos_totales"].sum() + proc["readecuadas_ley_439"].fillna(0).sum())
assert round(anterior * 100, 2) == 83.39 and round(proc["resueltas"].sum() / proc["ingresos_totales"].sum() * 100, 2) == 100.67
assert int(proc["flag_acumula_mora"].sum()) == 608 and int((proc["tasa_resolucion_con_readecuadas"] < 1).sum()) == 747
con_cr = proc[proc["ingresos_totales"] > 0]
np.testing.assert_allclose(con_cr["tasa_resolucion"].values, (con_cr["resueltas"] / con_cr["ingresos_totales"]).astype(float).values)
assert proc.loc[proc["ingresos_totales"] == 0, "tasa_resolucion"].isna().all()
assert (proc["tasa_resolucion"].dropna() >= 0).all()
assert (proc["tasa_pendencia"].dropna() <= 1.0001).all()
assert (proc["tasa_congestion"].dropna() >= 0.999).all()
con_tc = proc[proc["tasa_congestion"].notnull()]
assert (con_tc["tasa_congestion_winsorizada"] <= con_tc["tasa_congestion"] + 1e-6).all()
assert abs(spearmanr(proc["atendidas"], proc["log_atendidas"])[0] - 1) < 1e-4
assert len(pd.read_csv(CURATED / "eda" / "matriz_nulos_clasificada.csv")) == 87
for t in ["resumen_congestion_departamental", "resumen_congestion_materia", "resumen_congestion_ambito", "top_procesos_congestionados"]:
    assert len(pd.read_csv(REPORTS / "tables" / (t + ".csv"))) > 0
ok("indicadores: identidad de ingresos, tasas en rango, transformaciones monótonas")

OK indicadores: identidad de ingresos, tasas en rango, transformaciones monótonas


## 8. Ampliación: capítulo 9, serie histórica y diagnóstico (notebooks 14 a 16)

In [9]:
EDA = CURATED / "eda"
mat = pd.read_csv(EDA / "indicadores_capitulo9_materia.csv")
assert mat.groupby("ambito").size().to_dict() == {"capital": 13, "nacional": 13, "provincia": 12}
nac = mat[mat["ambito"] == "nacional"]
# Totales del cuadro 9.1.9 y, por otro camino, la serie histórica 9.1.12 del mismo año.
serie = tabla("causas_serie_historica")
s23 = serie[(serie["ambito"] == "nacional") & (serie["gestion"] == 2023)].iloc[0]
for col in ["pendientes_inicio", "ingresadas", "resueltas", "pendientes_fin"]:
    assert abs(int(nac[col].sum()) - int(s23[col])) <= 1, col
assert int(nac["atendidas"].sum()) == 738841 and int(s23["atendidas"]) in (738841, 738842)
por_tipo = nac.groupby("tipo_materia")["variacion_stock"].sum().to_dict()
assert por_tipo == {"no penal": -1379.0, "penal": 13134.0, "ejecución penal": 7359.0}
assert int(nac["variacion_stock"].sum()) == 19114
dep = pd.read_csv(EDA / "indicadores_capitulo9_departamento.csv")
assert len(dep) == 9 and int(dep["pendientes_fin"].sum()) == int(nac["pendientes_fin"].sum())
assert int(dep["ingresadas"].sum()) == int(nac["ingresadas"].sum())
cap = pd.read_csv(EDA / "capacidad_resoluciones_extra.csv")
todas = cap[cap["grupo_materia"] == "TODAS LAS MATERIAS"].iloc[0]
assert int(round(todas["resoluciones_extra_para_frenar"])) == 19115
assert int(round(todas["resoluciones_extra_para_vaciar_3_anios"])) == 137951
ok("capítulo 9: 13 grupos de materia, totales iguales a la serie 9.1.12, +19.114 de stock (no penal −1.379 · penal +13.134 · ejecución +7.359)")

OK capítulo 9: 13 grupos de materia, totales iguales a la serie 9.1.12, +19.114 de stock (no penal −1.379 · penal +13.134 · ejecución +7.359)


In [10]:
ind = pd.read_csv(EDA / "serie_historica_indicadores.csv")
assert len(ind) == 51 and sorted(ind["gestion"].unique()) == list(range(2007, 2024))
n23 = ind[(ind["ambito"] == "nacional") & (ind["gestion"] == 2023)].iloc[0]
assert round(n23["tasa_resolucion"], 2) == 0.95 and round(n23["tasa_congestion"], 2) == 1.93
enlace = pd.read_csv(EDA / "serie_enlace_stock.csv")
ne = enlace[enlace["ambito"] == "nacional"]
assert len(ne) == 16 and int((ne["diferencia"] < 0).sum()) == 15 and int(ne["diferencia"].sum()) == -546074
comp = pd.read_csv(EDA / "comparacion_ceja.csv")
assert len(comp) == 6 and comp.loc[comp["gestion"].between(2019, 2022), "dif_tr"].abs().max() <= 0.03
assert comp.loc[comp["gestion"] == 2023, "tr_ceja"].isna().all()
mat_s = pd.read_csv(EDA / "serie_resueltas_materia_2020_2023.csv")
assert len(mat_s) == 12 and int(mat_s["2023"].sum()) == int(s23["resueltas"])
ok("serie 2007-2023: 51 filas, 2023 con resolución 0,95 y congestión 1,93, apertura menor que el cierre previo en 15 de 16 años, comparación con el CEJA")

OK serie 2007-2023: 51 filas, 2023 con resolución 0,95 y congestión 1,93, apertura menor que el cierre previo en 15 de 16 años, comparación con el CEJA


In [11]:
conc = pd.read_csv(EDA / "concentracion_rezago.csv")
top100 = conc[(conc["nivel"] == "filas con más pendientes") & conc["categoria"].str.startswith("las 100")].iloc[0]
assert abs(top100["pct_del_rezago"] - 73.4) < 0.1
tipos = conc[conc["nivel"] == "tipo de proceso"]
assert len(tipos) == 87 and abs(tipos["pct_del_rezago"].head(10).sum() - 65.7) < 0.1
assert abs(conc[conc["nivel"] == "departamento"]["pct_del_rezago"].sum() - 100) < 1e-6
desc = pd.read_csv(EDA / "descomposicion_varianza_duracion.csv").set_index("factor")
for col in ["r2_ajustado_sin_pesos", "r2_ajustado_pond_atendidas", "r2_ajustado_pond_pendientes"]:
    assert desc.loc["ámbito (capital / provincia)", col] < 0.02
    assert desc.loc["tipo de proceso", col] > desc.loc["materia", col]
agr = pd.read_csv(EDA / "agrupamiento_prueba.csv")
assert len(agr) == 6 and agr["silhouette"].max() < 0.5 and int(agr.loc[agr["silhouette"].idxmax(), "k"]) == 3
ok("diagnóstico: 100 filas = 73,4 % del rezago, 10 tipos = 65,7 %, el ámbito no explica la duración y el tipo de proceso explica más que la materia, y el agrupamiento no supera 0,5 de silhouette")

OK diagnóstico: 100 filas = 73,4 % del rezago, 10 tipos = 65,7 %, el ámbito no explica la duración y el tipo de proceso explica más que la materia, y el agrupamiento no supera 0,5 de silhouette


In [12]:
asim = pd.read_csv(EDA / "asimetria_antes_despues.csv")
assert len(asim) == 5 and (asim["asimetria_log1p"].abs() < asim["asimetria_original"].abs()).all()
assert (asim.set_index("variable").loc[["atendidas", "resueltas", "pendientes_fin"], "lectura_log1p"] == "asimetría moderada").all()
seg = pd.read_csv(EDA / "segmentos_materia.csv")
assert len(seg) == 5 and int(seg["n_procesos"].sum()) == 1082
ctx = pd.read_csv(EDA / "contexto_grupo_procesos.csv")
assert len(ctx) == 1122 and int((ctx["razon_vs_mediana"] > 3).sum()) == 198
tmp = pd.read_csv(EDA / "serie_variables_temporales.csv")
assert len(tmp) == 17 and tmp.loc[tmp["salto_ingresadas"], "gestion"].tolist() == [2013, 2016, 2020, 2021]
cor = pd.read_csv(EDA / "correlaciones_indicadores.csv")
assert len(cor) == 28 and int(cor["|r| > 0,90"].sum()) == 7 and int((cor["|r| > 0,90"] & cor["ambas_son_cantidades_contables"]).sum()) == 4
pca = pd.read_csv(EDA / "pca_varianza.csv")
assert abs(pca["varianza_pct"].sum() - 100) < 1e-6 and abs(pca["acumulada_pct"][1] - 79.6) < 0.1 and abs(pca["acumulada_pct"][2] - 97.8) < 0.1
mv = pd.read_csv(EDA / "outliers_multivariados.csv")
assert len(mv) == 1082 and int(mv["es_outlier_multivariado"].sum()) == 20
assert pd.read_parquet(CURATED / "features" / "dataset_analitico_curado.parquet").shape[1] == 118, "el notebook 17 no debe modificar la capa curada"
ok("técnicas de clase: log1p baja la asimetría, 198 procesos a más de 3 veces la mediana de su grupo, 7 pares con |r| > 0,90 (4 contables), PCA 79,6 % con 2 componentes, 20 outliers multivariados, capa curada intacta")

OK técnicas de clase: log1p baja la asimetría, 198 procesos a más de 3 veces la mediana de su grupo, 7 pares con |r| > 0,90 (4 contables), PCA 79,6 % con 2 componentes, 20 outliers multivariados, capa curada intacta


In [13]:
rc = pd.read_csv(EDA / "recursos_carga_departamento.csv")
assert len(rc) == 9 and rc["departamento"].is_unique
assert int(rc["items_total"].sum()) == 6766 and int((rc["items_total"] - rc["items_ocupados"]).sum()) == 316 and int(rc["organos"].sum()) == 1209
assert int(rc["ingresadas"].sum()) == int(nac["ingresadas"].sum()) and int(rc["resueltas"].sum()) == int(nac["resueltas"].sum())
assert abs(rc["ingresadas_por_ocupado"].median() - 52.1) < 0.1 and rc.loc[rc["ingresadas_por_ocupado"].idxmax(), "departamento"] == "Santa Cruz"
assert rc.loc[rc["ingresadas_por_ocupado"].idxmin(), "departamento"] == "Pando"
rcor = pd.read_csv(EDA / "correlaciones_recursos_departamento.csv")
assert len(rcor) == 15 and int(rcor["significativo_al_5pct"].sum()) == 0
ok("recursos: 6.766 ítems (316 vacantes) y 1.209 órganos en 9 departamentos, movimiento igual al notebook 14, carga mediana de 52,1 causas por puesto ocupado y ninguna de las 15 relaciones con los resultados es significativa")

OK recursos: 6.766 ítems (316 vacantes) y 1.209 órganos en 9 departamentos, movimiento igual al notebook 14, carga mediana de 52,1 causas por puesto ocupado y ninguna de las 15 relaciones con los resultados es significativa


In [14]:
print(len(resultados), "bloques verificados: todo OK")

12 bloques verificados: todo OK
